# Task 04: Talking/Whispering Detection

Detects mouth movement. Rapid open-close = talking.

Run cells 1-4 in order. Press 'q' to quit.

In [1]:
# CELL 1: Load model
import cv2, time, numpy as np, mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from collections import deque

face_landmarker = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path='../01_head_direction/face_landmarker.task'),
    num_faces=1, min_face_detection_confidence=0.5, min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5, running_mode=vision.RunningMode.IMAGE))

print('Face Landmarker loaded!')

Face Landmarker loaded!


In [2]:
# CELL 2: Configuration

# Mouth open threshold — MAR (mouth aspect ratio) above this = mouth is open
MOUTH_OPEN_THRESHOLD = 0.15    # Try 0.1 (sensitive) to 0.2 (loose)

# How many open-close cycles in the window = talking?
TALKING_CYCLES = 3             # 3+ cycles in 2 seconds = talking

# Time window to count cycles (seconds)
CYCLE_WINDOW = 2.0

# Scoring
TALKING_WEIGHT = 50            # Max score for talking detection

print(f'Config: mouth threshold={MOUTH_OPEN_THRESHOLD}, cycles={TALKING_CYCLES} in {CYCLE_WINDOW}s')

Config: mouth threshold=0.15, cycles=3 in 2.0s


In [3]:
# CELL 3: Detection function

def get_mouth_ratio(fl, w, h):
    """
    Calculate Mouth Aspect Ratio (MAR).
    
    Key lip landmarks:
      13 = upper lip center (top)
      14 = lower lip center (bottom)
      78 = left mouth corner
      308 = right mouth corner
    
    MAR = vertical distance / horizontal distance
    Closed mouth: MAR ~ 0.05
    Open mouth: MAR ~ 0.3+
    """
    upper_lip = fl[13]   # Top center of upper lip
    lower_lip = fl[14]   # Bottom center of lower lip
    left_corner = fl[78]  # Left mouth corner
    right_corner = fl[308]  # Right mouth corner
    
    # Vertical distance (lip opening)
    vertical = abs(upper_lip.y - lower_lip.y) * h
    
    # Horizontal distance (mouth width)
    horizontal = abs(right_corner.x - left_corner.x) * w
    
    if horizontal == 0:
        return 0
    
    mar = vertical / horizontal
    return mar


class TalkingDetector:
    """Tracks mouth open/close cycles to detect talking."""
    
    def __init__(self):
        self.prev_open = False          # Was mouth open in previous frame?
        self.cycle_times = deque()      # Timestamps of open->close transitions
    
    def update(self, mar, current_time):
        """
        Feed new mouth ratio. Returns (is_talking, cycle_count, score).
        
        Logic:
        1. Check if mouth is currently open
        2. If it just CLOSED (was open, now closed) = one cycle complete
        3. Count cycles in the last CYCLE_WINDOW seconds
        4. If cycles >= TALKING_CYCLES = talking!
        """
        is_open = mar > MOUTH_OPEN_THRESHOLD
        
        # Detect close transition (was open, now closed = 1 cycle)
        if self.prev_open and not is_open:
            self.cycle_times.append(current_time)
        
        self.prev_open = is_open
        
        # Remove old cycles outside the window
        while self.cycle_times and (current_time - self.cycle_times[0]) > CYCLE_WINDOW:
            self.cycle_times.popleft()
        
        cycle_count = len(self.cycle_times)
        is_talking = cycle_count >= TALKING_CYCLES
        
        # Score scales with number of cycles
        if not is_talking:
            score = 0
        else:
            score = min(TALKING_WEIGHT, int((cycle_count / 8) * TALKING_WEIGHT))
        
        return is_talking, cycle_count, score, is_open


print('Talking detector ready!')
print('Tracks mouth open/close CYCLES — not just open mouth.')
print('Yawning (1 open) = OK. Talking (3+ open-close) = FLAGGED.')

Talking detector ready!
Tracks mouth open/close CYCLES — not just open mouth.
Yawning (1 open) = OK. Talking (3+ open-close) = FLAGGED.


In [ ]:
# CELL 4: Live detection
cam = cv2.VideoCapture(0)
cam.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cam.isOpened():
    print('Cannot open webcam!')
else:
    print('Talking Detection running!')
    print('Try: talk normally, whisper, yawn, stay silent.')
    print('Press q to quit.\n')
    
    detector = TalkingDetector()
    fc = 0; t0 = time.time()
    
    while True:
        ok, f = cam.read()
        if not ok: break
        f = cv2.flip(f, 1); fc += 1
        h, w = f.shape[:2]
        
        rgb = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        mi = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        fr = face_landmarker.detect(mi)
        
        if fr.face_landmarks and len(fr.face_landmarks) > 0:
            fl = fr.face_landmarks[0]
            
            # Get mouth ratio
            mar = get_mouth_ratio(fl, w, h)
            
            # Update talking detector
            is_talking, cycles, score, is_open = detector.update(mar, time.time())
            
            # Draw mouth landmarks
            for idx in [13, 14, 78, 308]:  # Upper lip, lower lip, corners
                px = int(fl[idx].x * w)
                py = int(fl[idx].y * h)
                cv2.circle(f, (px, py), 3, (0, 255, 0), -1)
            
            # Draw mouth outline
            mouth_pts = [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308,
                        324, 318, 402, 317, 14, 87, 178, 88, 95]
            for i in range(len(mouth_pts) - 1):
                p1 = (int(fl[mouth_pts[i]].x * w), int(fl[mouth_pts[i]].y * h))
                p2 = (int(fl[mouth_pts[i+1]].x * w), int(fl[mouth_pts[i+1]].y * h))
                color = (0, 0, 255) if is_talking else (0, 255, 0)
                cv2.line(f, p1, p2, color, 1)
            
            # Status bar
            if is_talking:
                bar_color = (0, 0, 255)  # Red
                status = f'TALKING DETECTED! Cycles:{cycles} Score:{score}/{TALKING_WEIGHT}'
            elif is_open:
                bar_color = (0, 180, 200)  # Yellow
                status = f'Mouth open (MAR:{mar:.2f}) Cycles:{cycles}'
            else:
                bar_color = (0, 180, 0)  # Green
                status = f'Silent (MAR:{mar:.2f}) Cycles:{cycles}'
            
            cv2.rectangle(f, (0, 0), (w, 40), bar_color, -1)
            cv2.putText(f, status, (10, 28),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)
            
            # MAR bar visualization (shows mouth openness)
            bar_h = int(mar * 300)  # Scale for visibility
            bar_x = w - 40
            cv2.rectangle(f, (bar_x, h - bar_h), (bar_x + 20, h), (0, 255, 255), -1)
            cv2.line(f, (bar_x - 5, h - int(MOUTH_OPEN_THRESHOLD * 300)),
                    (bar_x + 25, h - int(MOUTH_OPEN_THRESHOLD * 300)), (0, 0, 255), 2)
            cv2.putText(f, 'MAR', (bar_x - 5, h - int(MOUTH_OPEN_THRESHOLD * 300) - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.3, (200, 200, 200), 1)
            
            # Debug
            cv2.putText(f, f'MAR:{mar:.3f} Open:{is_open} Cycles:{cycles}',
                       (10, h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)
        else:
            cv2.rectangle(f, (0, 0), (w, 40), (100, 100, 100), -1)
            cv2.putText(f, 'No face detected', (10, 28),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        el = time.time() - t0
        fps = fc / el if el > 0 else 0
        cv2.putText(f, f'FPS:{fps:.0f}', (w - 80, h - 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, (150, 150, 150), 1)
        
        cv2.imshow('ExamGuard - Talking Detection (q=quit)', f)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
    
    cam.release()
    cv2.destroyAllWindows()
    print(f'Done! {el:.0f}s, {fps:.0f} FPS')

Talking Detection running!
Try: talk normally, whisper, yawn, stay silent.
Press q to quit.



## Test Scenarios

| You do | Expected |
|--------|----------|
| Stay silent, mouth closed | GREEN - Silent |
| Yawn (open mouth once, hold) | YELLOW - Mouth open, Cycles: 0 |
| Talk normally | RED - TALKING DETECTED, Cycles: 3+ |
| Whisper (small mouth movements) | RED - should catch if movements are enough |
| Chew gum | Might trigger - similar motion to talking |

### Key insight:
Single open mouth = NOT talking (could be yawning, breathing).
Rapid open-close CYCLES = talking.

### Tuning:
- Missing talking? Lower `MOUTH_OPEN_THRESHOLD` (e.g., 0.1)
- False alarms? Raise `MOUTH_OPEN_THRESHOLD` (e.g., 0.2) or raise `TALKING_CYCLES` (e.g., 4)

## Next Step

Task 05: Combine ALL detectors (head + eyes + body + talking) + web app